# Gradient checking (NumPy)

Backpropagation is easy to get subtly wrong. Gradient checking compares the analytic gradient with a numerical one
(central differences) and reports the relative difference. The checker is in `gradient_check.py`, the network in `model.py`,
and `bugs.py` contains backward passes with deliberate mistakes.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone -b feat/gradient-checking https://github.com/nyawmolny/deep-learning-fundamentals/

Cloning into 'deep-learning-fundamentals'...
remote: Enumerating objects: 371, done.
remote: Counting objects: 100% (371/371), done.
remote: Compressing objects: 100% (267/267), done.
remote: Total 371 (delta 90), reused 336 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (371/371), 4.67 MiB | 23.81 MiB/s, done.
Resolving deltas: 100% (90/90), done.


In [3]:
%cd deep-learning-fundamentals/02-regularization-and-optimization/gradient-checking/

/content/deep-learning-fundamentals/02-regularization-and-optimization/gradient-checking


In [4]:
!ls

bugs.py  gradient_check.py  model.py  test_gradient_check.py


In [ ]:
import numpy as np
from bugs import BUGS, broken_backward
from gradient_check import gradient_check, numerical_gradients, relative_difference
from model import (
    backward_propagation,
    cost_from_params,
    forward_propagation,
    initialize_parameters,
)
from sklearn.datasets import make_moons

## 1. One-dimensional example

J(theta) = theta * x has the gradient x. The numerical estimate must agree.

In [7]:
x, theta = 4.0, 3.0
params = {"theta": np.array([[theta]])}
numeric = numerical_gradients(lambda p: float(p["theta"][0, 0] * x), params)["dtheta"][0, 0]
print(f"analytic: {x} | numerical: {numeric:.10f} | relative difference: {relative_difference(numeric, x):.1e}")

analytic: 4.0 | numerical: 3.9999999935 | relative difference: 8.2e-10


## 2. A correct network passes

In [8]:
X, y = make_moons(20, noise=0.2, random_state=0)
X, Y = X.T, y.reshape(1, -1).astype(float)

params = initialize_parameters([2, 5, 3, 1], seed=1, random_biases=True)
AL, caches = forward_propagation(X, params)
grads = backward_propagation(AL, Y, caches)
cost = lambda p: cost_from_params(p, X, Y)

diff, per_param, ok = gradient_check(cost, params, grads)
print(f"overall relative difference: {diff:.1e} | passed: {ok}")
print({k: f"{v:.1e}" for k, v in per_param.items()})

overall relative difference: 2.9e-09 | passed: True
{'W1': '1.1e-08', 'b1': '3.7e-09', 'W2': '2.7e-09', 'b2': '1.4e-09', 'W3': '1.4e-09', 'b3': '1.6e-09'}


## 3. Deliberate bugs are caught, and the per-parameter differences show where

In [9]:
print(f"{'bug':<26} | overall diff | parameters above threshold")
for bug in BUGS:
    diff, per_param, ok = gradient_check(cost, params, broken_backward(AL, Y, caches, bug))
    bad = [k for k, v in per_param.items() if v > 1e-6]
    print(f"{bug:<26} | {diff:12.2e} | {', '.join(bad)}")

bug                        | overall diff | parameters above threshold
missing 1/m in dW          |     8.92e-01 | W1, W2, W3
missing ReLU mask          |     2.57e-01 | W1, b1, W2, b2
wrong bias gradient        |     6.35e-01 | b1, b2, b3
dropped last-layer term    |     3.71e-01 | W3


## 4. Results and notes

- **Correct network:** overall relative difference 2.9e-09 (passes, threshold 1e-6). **Bugs:** missing 1/m in dW gives 8.92e-01 (W1, W2, W3); missing ReLU mask gives 2.57e-01 (W1, b1, W2, b2); wrong bias gradient gives 6.35e-01 (b1, b2, b3); dropped last-layer term gives 3.71e-01 (W3). Every bug is 5–8 orders of magnitude worse than the correct pass, so all are caught unambiguously.
- **Per-parameter differences localize the bug:** each mistake flags exactly the parameters it touches (wrong sign flags only biases; missing ReLU mask flags hidden layers but not the output layer; dropped term flags only W3), which points to the cause, not just "somewhere in the backward pass".
- **When it is used:** as an offline debugging check of the backward pass, on a small network and a few examples, and it is switched off during training because it costs two forward passes per parameter element.